# 🛰️ Remote Sensing Change Detection — Google Colab T4 Eğitim Defteri

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/osmankalkan/remote-sensing-change-detection/blob/main/notebooks/colab_train.ipynb)

> **Hedef:** Google Colab ortamında (NVIDIA T4 GPU) depoyu klonlamak/güncellemek, Google Drive'ı bağlamak, gerekli kütüphaneleri kurmak, `src/train.py` eğitim scriptini arka planda çalıştırmak ve en iyi model ağırlıklarını (`best_model.pth`) doğrudan Google Drive'a kalıcı olarak kaydetmek.

### 📋 İş Akışı Adımları:
1. **GPU & Ortam Kontrolü:** NVIDIA T4 GPU ve CUDA doğrulaması.
2. **Google Drive Bağlantısı:** Ağırlıkların ve logların kaydedileceği kalıcı Drive dizini oluşturma.
3. **GitHub Depo Yönetimi:** Repoyu klonlama veya en son değişiklikleri `git pull` ile çekme.
4. **Bağımlılıkların Kurulumu:** `requirements.txt` ve ek kütüphanelerin yüklenmesi.
5. **Veri Seti Hazırlığı:** Drive'dan veri çıkarma veya hızlı test için sentetik LEVIR-CD yamaları oluşturma.
6. **Eğitim Konfigürasyonu:** T4 GPU için optimize edilmiş `config_colab.yaml` oluşturma.
7. **Arka Planda Eğitim (`nohup`):** `src/train.py` sürecini arka planda başlatma.
8. **Canlı İzleme & Metrik Takibi:** Canlı logları ve IoU/F1 değerlerini izleme (Hücre kesilse dahi eğitim devam eder).
9. **Google Drive Doğrulaması:** `best_model.pth` kontrolü ve doğrulama görselleştirmelerini inceleme.
10. **Yardımcı Araçlar:** Süreç yönetimi ve GPU bellek temizleme.

## 1. GPU ve Donanım Kontrolü
T4 GPU'nun bağlı olduğunu ve CUDA'nın aktif olduğunu doğrulayalım.
*(Colab Menü: **Çalışma Zamanı** -> **Çalışma zamanı türünü değiştir** -> **T4 GPU**)*

In [ ]:
import os
import torch
import subprocess

print("=" * 60)
print("🚀 DONANIM VE SİSTEM KONTROLÜ")
print("=" * 60)

cuda_ok = torch.cuda.is_available()
print(f"CUDA Kullanılabilir mi?: {cuda_ok}")

if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Aktif GPU: {gpu_name}")
    print(f"Toplam VRAM: {vram_gb:.2f} GB")
    if "T4" in gpu_name:
        print("✅ NVIDIA Tesla T4 GPU aktif! (T4 16GB VRAM change detection için idealdir)")
    else:
        print(f"ℹ️ {gpu_name} GPU aktif.")
else:
    print("⚠️ UYARI: GPU algılanamadı!")
    print("Lütfen Colab menüsünden: Runtime -> Change runtime type -> T4 GPU seçin.")

!nvidia-smi

## 2. Google Drive Bağlantısı (Mount)
Model ağırlıklarının (`best_model.pth`), eğitim loglarının ve doğrulama görsellerinin Colab oturumu kapansa dahi kaybolmaması için Google Drive'ı bağlıyoruz.

In [ ]:
from google.colab import drive
import os

print("📂 Google Drive bağlanıyor...")
drive.mount('/content/drive')

# Google Drive üzerinde projeye özel kalıcı klasör
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/remote_sensing_change_detection"
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, "checkpoints")
DRIVE_VIS_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, "visualizations")

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_VIS_DIR, exist_ok=True)

print("\n✅ Google Drive başarıyla bağlandı!")
print(f"📁 Model Ağırlıkları Kayıt Dizini: {DRIVE_CHECKPOINT_DIR}")
print(f"🖼️ Doğrulama Görselleri Dizini: {DRIVE_VIS_DIR}")

## 3. GitHub Deposunu Klonlama veya Güncelleme
Proje dizini mevcut değilse `git clone` yapılır, mevcutsa `git pull` ile en güncel kodlar çekilir.

In [ ]:
import os

REPO_URL = "https://github.com/osmankalkan/remote-sensing-change-detection.git"
LOCAL_REPO_DIR = "/content/remote-sensing-change-detection"
BRANCH = "main"

if os.path.exists(LOCAL_REPO_DIR):
    print(f"🔄 Depo zaten mevcut: {LOCAL_REPO_DIR}")
    print(f"📥 Güncellemeler alınıyor (git fetch & pull)...")
    %cd {LOCAL_REPO_DIR}
    !git fetch origin
    !git reset --hard origin/{BRANCH} || !git pull origin {BRANCH}
else:
    print(f"📥 Depo klonlanıyor: {REPO_URL}...")
    %cd /content
    !git clone {REPO_URL}
    %cd {LOCAL_REPO_DIR}

print("\n✅ Aktif Çalışma Dizini:")
%pwd

## 4. Gerekli Kütüphanelerin Kurulumu
`requirements.txt` dosyasındaki kütüphaneleri (PyTorch, Albumentations, segmentation-models-pytorch vb.) kuruyoruz.

In [ ]:
%cd {LOCAL_REPO_DIR}
print("📦 Kütüphaneler kuruluyor (requirements.txt)...")
!pip install -q -r requirements.txt
!pip install -q gdown huggingface_hub

# Kurulum Doğrulama
import segmentation_models_pytorch as smp
import albumentations as A
import cv2
import yaml

print("=" * 60)
print("✅ Kütüphane kurulumları başarıyla tamamlandı:")
print(f" - PyTorch: {torch.__version__} (CUDA: {torch.version.cuda})")
print(f" - Segmentation Models PyTorch: {smp.__version__}")
print(f" - Albumentations: {A.__version__}")
print(f" - OpenCV: {cv2.__version__}")
print("=" * 60)

## 5. Veri Seti Hazırlığı & Kontrolü
Model eğitimi için `data/processed/{train, val}/` altında 256x256 yamalanmış görüntüler gereklidir.
- **Seçenek A:** Google Drive'ınızda hazır veri seti zip'iniz varsa otomatik olarak Colab yerel SSD'sine açılır.
- **Seçenek B:** Henüz veri seti yüklemediyseniz, kod yapısını ve eğitimi test etmek için sentetik 1024x1024 çiftler üretilir ve 256x256 yamalanır.

In [ ]:
import os
import shutil

PROCESSED_TRAIN_A = "data/processed/train/A"
PROCESSED_VAL_A = "data/processed/val/A"

train_ready = os.path.exists(PROCESSED_TRAIN_A) and len(os.listdir(PROCESSED_TRAIN_A)) > 0

if not train_ready:
    print("ℹ️ data/processed dizininde hazır veri seti bulunamadı.")
    
    # Google Drive'da veri arşivi var mı? (örneğin levir_cd_processed.zip veya dataset.zip)
    drive_candidates = [
        os.path.join(DRIVE_PROJECT_DIR, "levir_cd_processed.zip"),
        os.path.join(DRIVE_PROJECT_DIR, "data_processed.zip"),
        os.path.join(DRIVE_PROJECT_DIR, "data.zip"),
    ]
    
    found_zip = None
    for cand in drive_candidates:
        if os.path.exists(cand):
            found_zip = cand
            break
            
    if found_zip:
        print(f"📦 Google Drive'da veri arşivi tespit edildi: {found_zip}")
        print("⚡ Hızlı disk erişimi için Colab yerel NVMe SSD'sine açılıyor...")
        !unzip -q "{found_zip}" -d data/
    else:
        print("🧪 Google Drive'da hazır arşiv bulunamadı.")
        print("🚀 Hızlı eğitim testi için sentetik bitemporal veri çiftleri üretiliyor...")
        !python scripts/prepare_sample_data.py --train_count 8 --val_count 4 --test_count 2
        print("✂️ 256x256 yamalama (patching) yapılıyor...")
        !python scripts/patch_images.py --raw_dir data/raw --out_dir data/processed --patch_size 256 --stride 256

train_count = len(os.listdir("data/processed/train/A")) if os.path.exists("data/processed/train/A") else 0
val_count = len(os.listdir("data/processed/val/A")) if os.path.exists("data/processed/val/A") else 0
print(f"\n✅ Veri Seti Hazır: {train_count} eğitim yaması | {val_count} doğrulama yaması")

## 6. Colab T4 GPU Eğitim Konfigürasyonu
Google Drive kayıt dizini (`save_dir`), batch size, epoch sayısı ve CUDA cihazı Colab T4 için ayarlanır.

In [ ]:
import yaml
import os

with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Colab T4 GPU Ayarları
config["training"]["device"] = "cuda"
config["training"]["batch_size"] = 8          # T4 GPU için 8 veya 16 önerilir
config["training"]["epochs"] = 30
config["training"]["save_dir"] = DRIVE_CHECKPOINT_DIR
config["data"]["num_workers"] = 2

colab_config_path = "configs/config_colab.yaml"
with open(colab_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Colab konfigürasyonu kaydedildi: {colab_config_path}")
print(f" - Model Tipi: {config['model']['type']} ({config['model']['encoder_name']})")
print(f" - Batch Size: {config['training']['batch_size']}")
print(f" - Epoch Sayısı: {config['training']['epochs']}")
print(f" - Kayıt Dizini (Google Drive): {config['training']['save_dir']}")

## 7. `src/train.py` Eğitim Scriptini Arka Planda Başlatma
Eğitim süreci `nohup` ve `&` ile arka planda çalıştırılır.
Bu sayede:
- Tarayıcı sekmesi kazara kapansa veya bağlantı kopsa bile eğitim Colab VM üzerinde kesintisiz devam eder.
- Çıktılar hem yerel `train.log` dosyasına yazılır hem de Colab konsolundan takip edilebilir.
- En iyi model ağırlıkları (`best_model.pth`) her IoU rekorunda doğrudan Google Drive'a kaydedilir.

In [ ]:
import subprocess
import time
import os

log_file_path = "/content/remote-sensing-change-detection/train.log"

# Varsa önceki eğitim logunu temizle
if os.path.exists(log_file_path):
    os.remove(log_file_path)

# Arka plan çalıştırma komutu (unbuffered Python ile anlık log akışı)
cmd = (
    f"nohup python -u src/train.py "
    f"--config configs/config_colab.yaml "
    f"--save_dir '{DRIVE_CHECKPOINT_DIR}' "
    f"> '{log_file_path}' 2>&1 &"
)

print("🚀 Eğitim arka planda başlatılıyor...")
os.system(cmd)
time.sleep(3)

# Süreç kontrolü
ps_output = subprocess.getoutput("ps aux | grep 'src/train.py' | grep -v 'grep'")
if ps_output:
    print("=" * 65)
    print("✅ EĞİTİM ARKA PLANDA BAŞARIYLA ÇALIŞIYOR!")
    print("=" * 65)
    print(f"Süreç Bilgisi:\n{ps_output.strip()}")
    print(f"\n📝 Canlı Log Dosyası: {log_file_path}")
    print(f"💾 Hedef Checkpoint: {DRIVE_CHECKPOINT_DIR}/best_model.pth")
else:
    print("⚠️ Süreç başlatılamadı. Hata detayları:")
    if os.path.exists(log_file_path):
        with open(log_file_path, "r") as f:
            print(f.read())

## 8. Canlı Eğitim Takibi & Metrik Monitörü
Aşağıdaki hücre arka plandaki `train.log` dosyasını canlı olarak okur ve ekrana basar.
> 💡 **Önemli:** Bu hücreyi durdurmak (⏹️ Interrupt Kernel) arka plandaki eğitimi **durdurmaz**. İstediğiniz zaman hücreyi durdurup tekrar çalıştırabilirsiniz.

In [ ]:
import time
import os
import subprocess

print("👀 Canlı Log Akışı Başlatıldı (Durdurmak için hücreyi 'Interrupt' edin)\n" + "-" * 60)

log_file_path = "/content/remote-sensing-change-detection/train.log"

# Log dosyasının oluşmasını bekle
for _ in range(10):
    if os.path.exists(log_file_path) and os.path.getsize(log_file_path) > 0:
        break
    time.sleep(1)

try:
    with open(log_file_path, "r") as f:
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                # Süreç hala çalışıyor mu?
                is_running = subprocess.getoutput("pgrep -f 'src/train.py'")
                if not is_running:
                    print("\n" + "=" * 60)
                    print("🏁 Eğitim süreci tamamlandı!")
                    print("=" * 60)
                    break
                time.sleep(1)
except KeyboardInterrupt:
    print("\n\n⏸️ Canlı log izleme kullanıcı tarafından durduruldu.")
    print("ℹ️ Eğitim arka planda kesintisiz devam ediyor.")

## 9. Google Drive Doğrulaması & Kaydedilen Model Ağırlıkları
Google Drive'daki `best_model.pth` dosyasını inceler, ulaşılan en iyi IoU / F1 skorlarını ve görselleştirme çıktılarını gösterir.

In [ ]:
import os
import torch
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

best_model_path = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pth")
latest_model_path = os.path.join(DRIVE_CHECKPOINT_DIR, "latest_model.pth")

print("=" * 65)
print("🔍 GOOGLE DRIVE CHECKPOINT RAPORU")
print("=" * 65)

if os.path.exists(best_model_path):
    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
    print(f"✅ En İyi Model: {best_model_path} ({size_mb:.2f} MB)")
    
    ckpt = torch.load(best_model_path, map_location="cpu")
    print(f"📌 En İyi Epoch: {ckpt.get('epoch', 'Bilinmiyor')}")
    scores = ckpt.get("val_scores", {})
    print(f"🎯 Val IoU:        {scores.get('iou', 0):.4f}")
    print(f"🎯 Val F1-Skor:    {scores.get('f1', 0):.4f}")
    print(f"🎯 Val Hassasiyet: {scores.get('precision', 0):.4f}")
    print(f"🎯 Val Duyarlılık: {scores.get('recall', 0):.4f}")
else:
    print("⏳ best_model.pth henüz kaydedilmedi (Eğitim henüz ilk epochlarda olabilir).")

# Varsa görselleştirme snapshot'larını göster
vis_dir = os.path.join(DRIVE_CHECKPOINT_DIR, "visualizations")
vis_images = sorted(glob.glob(os.path.join(vis_dir, "*.png")))

if vis_images:
    latest_img_path = vis_images[-1]
    print(f"\n🖼️ Son Doğrulama Görselleştirmesi: {latest_img_path}")
    img = mpimg.imread(latest_img_path)
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Doğrulama Tahmini - {os.path.basename(latest_img_path)}", fontsize=12, weight='bold')
    plt.show()
else:
    print(f"ℹ️ Henüz görselleştirme üretilmedi (Her 5 epochta bir kaydedilir).")

## 10. Yardımcı Araçlar (Süreç Durdurma & GPU Belleği Temizleme)
Gerektiğinde arka plandaki eğitimi durdurmak veya GPU önbelleğini temizlemek için bu hücreyi kullanabilirsiniz.

In [ ]:
# 🛑 Arka plandaki eğitimi durdurmak için aşağıdaki satırın yorumunu kaldırıp çalıştırın:
# !pkill -f 'src/train.py'
# print("Eğitim süreci durduruldu.")

# 🧹 GPU Önbelleğini Temizlemek İsterseniz:
# import torch
# torch.cuda.empty_cache()
# print("GPU önbelleği boşaltıldı.")